# 스케치 → 광고 이미지 PoC (광고 이미지 만들기 4번)

사장님이 그린 구도를 따라 광고 이미지를 만들 수 있는지 확인한다.

**처음 가정은 틀렸다.** ControlNet·IP-Adapter 가 SD1.5·SDXL 생태계에 몰려 있어서
귀한님이 쓰는 `sd-turbo`(SD 2.1 계열) 로는 안 될 줄 알았고, 팀 전체가 모델을
바꿔야 하는 줄 알았다. 실험해보니 **SD 2.1 용 ControlNet 으로 그대로 된다.**

GPU: RTX 3060 Ti 8GB / Colab T4 에서도 동일하게 확인.

In [ ]:
!pip install -q diffusers transformers accelerate

## 1. 파이프라인

`sd-turbo` 는 SD 2.1 계열이라 SD1.5용 ControlNet 이 안 맞는다. 2.1용을 쓴다.

In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline

controlnet = ControlNetModel.from_pretrained(
    "thibaud/controlnet-sd21-scribble-diffusers", torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stabilityai/sd-turbo",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

# 모델이 한국어를 모른다 (prompt_builder 가 영어로 바꾸는 이유와 같다)
PROMPT = "appetizing fried chicken on a plate, warm indoor lighting, food photography"

## 2. 스케치가 실제로 반영되는가 — 대조 실험

⚠️ **한 장만 보고 판단하면 안 된다.** 음식 사진에는 접시가 원래 나오므로
"접시가 있네" 는 근거가 못 된다. 구도를 서로 다르게 준 넷을 같은 seed 로
돌려서, 결과가 스케치를 따라 갈리는지를 본다. **빈 그림이 기준선이다.**

In [ ]:
from PIL import Image, ImageDraw


def canvas():
    return Image.new("RGB", (512, 512), "black")


a = canvas(); ImageDraw.Draw(a).ellipse([40, 300, 300, 470], outline="white", width=8)
b = canvas(); ImageDraw.Draw(b).ellipse([300, 50, 460, 180], outline="white", width=8)
c = canvas()
for x in (60, 210, 360):
    ImageDraw.Draw(c).ellipse([x, 220, x + 100, 300], outline="white", width=8)

tests = [("왼쪽아래", a), ("오른쪽위", b), ("세개", c), ("빈것", canvas())]


def run(image, steps, scale, seed=0):
    return pipe(
        prompt=PROMPT,
        image=image,
        num_inference_steps=steps,
        guidance_scale=0.0,  # turbo 계열은 guidance 를 쓰지 않는다
        controlnet_conditioning_scale=scale,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]


def strip(images):
    out = Image.new("RGB", (512 * len(images), 512))
    for i, img in enumerate(images):
        out.paste(img, (512 * i, 0))
    return out


strip([run(s, steps=8, scale=1.5) for _, s in tests])

**결과: 스케치를 정확히 따라간다.**

| 스케치 | 결과 |
|---|---|
| 왼쪽 아래 접시 하나 | 왼쪽 아래에 접시 |
| 오른쪽 위 접시 하나 | 오른쪽 위에 접시 |
| 가로로 세 개 | 가로로 세 개 |
| **빈 것** | **접시 없음** |

빈 그림에 접시가 안 생긴 것이 결정적이다 — 우연이 아니라 스케치가 그림을 정했다.

다만 이 설정(8스텝·1.5)은 그림이 점묘화처럼 뭉개진다. 구조는 잡혔으니 품질을 찾는다.

## 3. 스텝과 세기 — 구조는 지키면서 안 뭉개지는 값 찾기

In [ ]:
combos = [(2, 1.0), (2, 1.3), (4, 1.0), (4, 1.3), (4, 1.6)]
print(combos)
strip([run(c, steps=s, scale=v) for s, v in combos])

**스텝을 늘릴수록 나빠진다.** `sd-turbo` 는 1~4스텝용으로 만든 모델이라
많이 돌리면 오히려 뭉개진다. 세기를 올려도 구조는 그대로인데 그림만 망가진다.

→ 적은 스텝 · 낮은 세기가 낫다.

## 4. 진짜 손그림 — 뒤집어야 하는가

지금까지는 검은 배경에 흰 선을 넣었다. 사장님이 올릴 건 **흰 종이에 검은 연필선**
이라 정반대다. ControlNet scribble 은 보통 흰 선을 기대하므로 뒤집어야 할 것으로
봤는데, 실제로 그런지 확인한다.

In [ ]:
from PIL import ImageOps

paper = Image.new("RGB", (512, 512), "white")
for x in (60, 210, 360):
    ImageDraw.Draw(paper).ellipse([x, 220, x + 100, 300], outline="black", width=8)

print("안뒤집음-1, 안뒤집음-2, 뒤집음-1, 뒤집음-2")
strip([
    run(s, steps=n, scale=1.0)
    for s in (paper, ImageOps.invert(paper))
    for n in (1, 2)
])

## 결론 — src/app_core/sketch_gen.py 로 옮긴 값

| | 값 | 이유 |
|---|---|---|
| 모델 | `stabilityai/sd-turbo` | **바꿀 필요 없었다.** 귀한님과 통일 유지 |
| ControlNet | `thibaud/controlnet-sd21-scribble-diffusers` | SD1.5용은 2.1 계열에 안 맞는다 |
| 스텝 | **1** | 늘리면 뭉개진다. 1스텝도 구조를 지킨다 (약 11 it/s) |
| conditioning | **1.0** | 올리면 구조는 그대로인데 그림이 망가진다 |
| 전처리 | **없음** | 뒤집으나 안 뒤집으나 똑같이 동작했다 |

### 남은 것

- 여기 스케치는 코드로 그린 깨끗한 선이다. **실제 종이 사진은 그림자·회색조·
  기울어짐이 섞인다** — 그 입력으로 다시 확인해야 한다.
- 한국어 주문 → 영어 프롬프트 변환은 귀한님 `prompt_builder` 를 쓴다 (#12 머지 후).